# SimPy を使った M/M/1 待ち行列の練習

## M/M/1 モデル

* $\lambda$: 平均到着率
    * 単位時間あたりに到着する客の人数
* $\mu$: 平均サービス率
    * 単位時間あたりに処理できる客の人数

平均待ち時間 $T_1$ は次式で与えられることが知られている。
$$
T_1 = \frac{\rho}{1-\rho} \mu
$$
$\rho$ は利用率($\lambda/\mu$。)

## シミュレーション

In [ ]:
import simpy
import random

In [ ]:
class Model():
    def __init__(self, env, customers_num, avg_arrival_rate, avg_service_rate):
        '''
        Model の初期化を行う。
        :param env                : SimPy の env
        :param customers_num      : 全客数
        :param avg_arrival_rate   : 1時間あたりに到着する客の数(平均到着率λ)
        :param avg_service_rate   : 1時間あたりに処理できる客の数(平均サービス率μ)
        '''
        self.env              = env
        self.customers_num    = customers_num
        self.avg_arrival_rate = avg_arrival_rate
        self.avg_service_rate = avg_service_rate
        self.wait_times       = []                # 待ち時間の記録先
        self.queue            = simpy.Store(env)  # 待ち行列

    def arrival_process(self):
        '''客の到着プロセス'''
        for counter in range(self.customers_num):
            # 指数分布に従う乱数秒待つ
            wait_time = random.expovariate(self.avg_arrival_rate) 
            yield self.env.timeout(wait_time)
            now = self.env.now
            #print(f"customer {counter} arrived at {now}")
            yield self.queue.put(now)

    def service_process(self):
        '''サービスのプロセス'''
        while True:
            arrival_time = yield self.queue.get() # 客が来るまで待つ
            now = self.env.now
            wait = now - arrival_time
            self.wait_times.append(wait)
            #print(f"t={now:.3f} Service Started (wait time={wait:.3f})")
            wait_time = random.expovariate(self.avg_service_rate)
            # 指数分布に従う乱数秒待つ
            yield self.env.timeout(wait_time)


In [ ]:
AVG_ARRIVAL_RATE = 10   # λ: 10人/時間
AVG_SERVICE_RATE = 12  # μ: 12人/時間
CUSTOMERS_NUM    = 100000

env = simpy.Environment()
model = Model(env, CUSTOMERS_NUM, AVG_ARRIVAL_RATE, AVG_SERVICE_RATE)
env.process(model.arrival_process())
env.process(model.service_process())

env.run()

print(f"平均到着率: {AVG_ARRIVAL_RATE}")
print(f"平均サービス率: {AVG_SERVICE_RATE}")
print(f"全客数: {CUSTOMERS_NUM}")
print(f"平均待ち時間: {sum(model.wait_times)/len(model.wait_times):.4f} 時間。")

In [ ]:
rho = AVG_ARRIVAL_RATE / AVG_SERVICE_RATE
avg_service_time = 1 / AVG_SERVICE_RATE          # 1/μ
theoretical_avg_wait_time = rho/(1-rho) * avg_service_time
print(f"平均待ち時間(理論): {theoretical_avg_wait_time:.3f} 時間。")